In [6]:
import pandas as pd
import numpy as np
train= pd.read_csv("crime_train.csv")
test= pd.read_csv("crime_test.csv")
print(train.shape, test.shape)
train.head()

(11771, 12) (9639, 12)


,Unnamed: 0,Num,case_filed,city,area,crime_description,age,sex,weapon,domain,police_department,closed
0,19794,19795,04-04-2022 18:00,Mumbai,307.0,VEHICLE - STOLEN,63.0,X,Poison,Violent Crime,12.0,No
1,13762,13763,07-27-2021 10:00,Surat,504.0,ROBBERY,57.0,F,Explosives,Violent Crime,3.0,No
2,30311,30312,06-16-2023 23:00,Thane,493.0,ASSAULT,33.0,F,Firearm,Violent Crime,18.0,No
3,5043,5044,07-29-2020 03:00,Ahmedabad,169.0,PUBLIC INTOXICATION,76.0,F,Knife,Other Crime,3.0,No
4,35008,35009,12-29-2023 16:00,Mumbai,133.0,PUBLIC INTOXICATION,13.0,M,NaN,Other Crime,13.0,No


In [3]:
print(train.isnull().sum())
print(train["closed"].value_counts())
print(test.columns.tolist())

Unnamed: 0              0
Num                     0
case_filed              1
city                    1
area                    1
crime_description       1
age                     1
sex                     1
weapon               1753
domain                  1
police_department       1
closed                  1
dtype: int64
closed
No     5942
Yes    5828
Name: count, dtype: int64
['Unnamed: 0', 'Num', 'case_filed', 'city', 'area', 'crime_description', 'age', 'sex', 'weapon', 'domain', 'police_department', 'closed']


In [7]:
train = train.dropna(subset=['closed'])
train = train.drop(columns=['Unnamed: 0', 'Num'], errors='ignore')

test = test.dropna(subset=['closed'])
test = test.drop(columns=['Unnamed: 0', 'Num'], errors='ignore')

for df in (train, test):
    df["weapon"] = df["weapon"].fillna("Unknown")
    d = pd.to_datetime(df["case_filed"], errors="coerce")
    df["hour"] = d.dt.hour
    df["month"] = d.dt.month
    df["year"] = d.dt.year
    df.drop(columns=["case_filed"], inplace=True)
    df["area"] = df["area"].astype(str)
    df["police_department"] = df["police_department"].astype(str)

    print(train.shape, test.shape) 


(11770, 12) (9639, 10)
(11770, 12) (9639, 12)


In [8]:

y_train = (train["closed"] == "Yes").astype(int).values
X_train = train.drop(columns=["closed"], errors="ignore")

has_labels = "closed" in test.columns
if has_labels:
    y_test = (test["closed"] == "Yes").astype(int).values
    X_test = test.drop(columns=["closed"], errors="ignore")
else:
    y_test = None
    X_test = test.copy()


In [9]:
import numpy as np


num_cols = ["age", "hour", "month", "year"]
cat_cols = [
    "city",
    "area",
    "crime_description",
    "sex",
    "weapon",
    "domain",
    "police_department",
]

In [8]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def train_model(X, y, lr=0.1, epochs=1000, lam=0.01):
    w = np.zeros(X.shape[1])
    for i in range(epochs):
        p = sigmoid(X @ w)                       
        grad = X.T @ (p - y) / len(y)           
        grad[1:] += lam * w[1:]                  
        w -= lr * grad                          
    return w

def predict(X, w):
    return (sigmoid(X @ w) >= 0.5).astype(int)

In [19]:
import numpy as np


X = X_train.values if hasattr(X_train, "values") else X_train
y = y_train.values if hasattr(y_train, "values") else y_train

np.random.seed(42)
import numpy as np


def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))


def predict(X, w):
    return (sigmoid(X @ w) >= 0.5).astype(int)


def train_model(X, y, lr=0.1, epochs=1000, lam=0.01):
    w = np.zeros(X.shape[1])
    n = len(y)

    for _ in range(epochs):
        p = sigmoid(X @ w)
        grad = (X.T @ (p - y)) / n
        grad[1:] += (lam * w[1:]) / n  # L2 regularization
        w -= lr * grad

    return w


In [20]:
import numpy as np
import pandas as pd

# 1. Convert categorical variables to numeric (One-Hot Encoding) & fill NaNs
X_features = pd.get_dummies(X_train[num_cols + cat_cols], drop_first=True)
X_features = X_features.fillna(0)

# 2. Convert X and y to floating-point NumPy arrays
X = X_features.values.astype(float)
y = y_train.astype(float)

# 3. Add a bias column (column of 1s) to X
X = np.hstack([np.ones((X.shape[0], 1)), X])

# 4. Shuffle and split into Train / Validation
np.random.seed(42)
idx = np.random.permutation(len(X))
cut = int(0.8 * len(idx))
tr_i, val_i = idx[:cut], idx[cut:]

X_tr, y_tr = X[tr_i], y[tr_i]
X_val, y_val = X[val_i], y[val_i]

# 5. Train model and evaluate
w = train_model(X_tr, y_tr)
pred = predict(X_val, w)

print("Validation accuracy:", (pred == y_val).mean())


Validation accuracy: 0.5093457943925234


In [21]:
import numpy as np
import pandas as pd

# 1. Align test features with training features
X_tr_features = pd.get_dummies(X_train[num_cols + cat_cols], drop_first=True)
X_te_features = pd.get_dummies(X_test[num_cols + cat_cols], drop_first=True)

# Reindex test columns so they match training columns exactly
X_te_features = X_te_features.reindex(columns=X_tr_features.columns, fill_value=0)

# 2. Standardize numerical features (Mean = 0, Std = 1) to improve accuracy
mean = X_tr_features.mean()
std = X_tr_features.std().replace(0, 1)  # avoid division by zero

X_tr_scaled = (X_tr_features - mean) / std
X_te_scaled = (X_te_features - mean) / std

# 3. Add bias term (column of 1s)
X_train_final = np.hstack([np.ones((X_tr_scaled.shape[0], 1)), X_tr_scaled.values])
X_test_final = np.hstack([np.ones((X_te_scaled.shape[0], 1)), X_te_scaled.values])
y_train_final = y_train.astype(float)

# 4. Retrain model on full scaled training dataset
w_final = train_model(X_train_final, y_train_final, lr=0.1, epochs=2000)

# 5. Make predictions on the test set
test_preds = predict(X_test_final, w_final)

# 6. Save results to a CSV file
submission = pd.DataFrame({
    'closed': np.where(test_preds == 1, 'Yes', 'No')
})
submission.to_csv("submission.csv", index=False)

print("Saved predictions to submission.csv successfully!")

Saved predictions to submission.csv successfully!


In [22]:
import pandas as pd

df_sub = pd.read_csv("submission.csv")
print("Shape:", df_sub.shape)
print(df_sub.head())
print("\nPrediction distribution:")
print(df_sub['closed'].value_counts())

Shape: (9639, 1)
  closed
0     No
1     No
2     No
3    Yes
4    Yes

Prediction distribution:
closed
No     6329
Yes    3310
Name: count, dtype: int64


In [23]:
import numpy as np

# 1. Add bias column to scaled training features
X_scaled_all = np.hstack([np.ones((X_tr_scaled.shape[0], 1)), X_tr_scaled.values])
y_all = y_train.astype(float)

# 2. Split scaled dataset
np.random.seed(42)
idx = np.random.permutation(len(X_scaled_all))
cut = int(0.8 * len(idx))
tr_i, val_i = idx[:cut], idx[cut:]

X_tr_s, y_tr_s = X_scaled_all[tr_i], y_all[tr_i]
X_val_s, y_val_s = X_scaled_all[val_i], y_all[val_i]

# 3. Train and check new validation accuracy
w_scaled = train_model(X_tr_s, y_tr_s, lr=0.1, epochs=2000)
val_pred = predict(X_val_s, w_scaled)

print("Scaled Validation Accuracy:", (val_pred == y_val_s).mean())

Scaled Validation Accuracy: 0.5174171622769753
